# Silver Layer: Deduplication and SCD2

## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from silver_config import SILVER_TABLES, DEDUP_CONFIG, SILVER_SCHEMA

## Load all Tables

In [0]:
silver_dfs = [spark.read.table(f"{SILVER_SCHEMA}.{silver_table}") for silver_table in SILVER_TABLES]

In [0]:
all_listings = silver_dfs[0]
for df in silver_dfs[1:]:
    all_listings = all_listings.unionByName(df)

In [0]:
null_price_listings = all_listings.filter(F.col("price_amount").isNull())
all_listings = all_listings.filter(F.col("price_amount").isNotNull())

## Deduplication

In [0]:
w = Window.partitionBy(*DEDUP_CONFIG["group_keys"][:1]).orderBy("price_amount").rangeBetween(
    -DEDUP_CONFIG["price_range"], DEDUP_CONFIG["price_range"]
)

all_listings = all_listings.withColumn(
    "nearby", F.collect_list(F.struct("listing_id", "description")).over(w)
)

all_listings = all_listings.withColumn(
    "nearby_others",
    F.filter("nearby", lambda x: x["listing_id"] != F.col("listing_id"))
)

all_listings = all_listings.withColumn(
    "distances",
    F.transform("nearby_others", lambda x: F.levenshtein(F.col("description"), x["description"]))
)

all_listings = all_listings.withColumn("min_distance", F.array_min("distances"))

duplicates = all_listings.filter(F.col("min_distance") < 15)

In [0]:
w_rank = Window.partitionBy("location_city", "price_amount").orderBy("listing_id")

duplicates = duplicates.withColumn("dedup_rank", F.row_number().over(w_rank))

urls_to_remove = duplicates.filter(F.col("dedup_rank") != 1).select("source_url")

all_listings = all_listings.join(urls_to_remove, on="source_url", how="left_anti")

In [0]:
all_listings = all_listings.drop(
    "nearby", "nearby_others", "distances", "candidates", "best_match",
    "min_distance", "matched_listing_id", "dedup_rank"
)
all_listings = all_listings.unionByName(null_price_listings)

## SCD2

In [0]:
new_listings = all_listings.select(
    "listing_id", "title", "price_amount", "currency", "rooms_count", "area",
    "location_city", "location_zip", "property_type", "transaction_type",
    "images", "description", "source", "source_url", "scraped_at", "posted_date"
)
new_listings.createOrReplaceTempView("new_listings")

In [0]:
%sql
MERGE INTO silver.listings AS target
USING (
    SELECT listing_id AS merge_key, * FROM new_listings

    UNION ALL

    SELECT NULL AS merge_key, updates.*
    FROM new_listings AS updates
    JOIN silver.listings AS target
    ON updates.listing_id = target.listing_id
    WHERE target.dwh_is_current = true
    AND updates.price_amount <> target.price_amount
) AS staged

ON target.listing_id = staged.merge_key AND target.dwh_is_current = true

WHEN MATCHED AND target.price_amount <> staged.price_amount THEN
    UPDATE SET target.dwh_is_current = false, target.dwh_valid_to = current_date()

WHEN NOT MATCHED THEN
    INSERT (listing_id, title, price_amount, currency, rooms_count, area,
            location_city, location_zip, property_type, transaction_type,
            images, description, source, source_url, scraped_at, posted_date,
            dwh_is_current, dwh_valid_from, dwh_valid_to)
    VALUES (staged.listing_id, staged.title, staged.price_amount, staged.currency, staged.rooms_count, staged.area,
            staged.location_city, staged.location_zip, staged.property_type, staged.transaction_type,
            staged.images, staged.description, staged.source, staged.source_url, staged.scraped_at, staged.posted_date,
            true, current_date(), null)